In [1]:
import os
import re
import copy
import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torch.multiprocessing as mp
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

mp.set_sharing_strategy("file_system")

# =========================
# Configuration
# =========================
# BASE_DIR = "/mnt/erencem-ozbey/ber_estimation"
BASE_DIR = "./"
DATA_PATH = os.path.join(BASE_DIR, "data_physics_with_variances_total.csv")

BEST_MODEL_SAVE_PATH = os.path.join(
    BASE_DIR, "ber_multiscale_rescnn_moe_best_stable_large_scaled_means_var_z01_5M.pth"
)
LAST_MODEL_SAVE_PATH = os.path.join(
    BASE_DIR, "ber_multiscale_rescnn_moe_last_stable_large_scaled_means_var_z01_5M.pth"
)
SCALER_SAVE_PATH = os.path.join(
    BASE_DIR, "ber_multiscale_rescnn_moe_scalers_large_scaled_means_var_z01_5M.pkl"
)

EPS = 1e-12


# =========================
# Utilities
# =========================
def get_sorted_seq_cols(columns, prefix):
    pattern = re.compile(rf"^{re.escape(prefix)}_(\d+)$")
    matched = []
    for col in columns:
        m = pattern.match(col)
        if m:
            matched.append((int(m.group(1)), col))
    matched.sort(key=lambda x: x[0])
    return [col for _, col in matched]


def make_strat_bins(y_log, n_bins=10):
    """
    Build stratification bins from log10(BER).
    Falls back safely if quantile edges collapse.
    """
    y_flat = y_log.reshape(-1)
    quantiles = np.linspace(0, 1, n_bins + 1)
    edges = np.quantile(y_flat, quantiles)
    edges = np.unique(edges)

    if len(edges) < 3:
        return None

    bins = np.digitize(y_flat, edges[1:-1], right=True)
    counts = np.bincount(bins)
    if np.any(counts < 2):
        return None
    return bins


def has_nonfinite_tensor(x):
    return not torch.isfinite(x).all().item()


def log10ber_to_regime_index(y_log, low_thr=1e-6, mid_thr=1e-3):
    """
    Maps target log10(BER) to:
      0 -> low BER: y < 1e-6
      1 -> mid BER: 1e-6 <= y < 1e-3
      2 -> high BER: y >= 1e-3
    """
    low_log = np.log10(low_thr)
    mid_log = np.log10(mid_thr)

    if isinstance(y_log, np.ndarray):
        out = np.full_like(y_log, fill_value=2, dtype=np.int64)
        out[y_log < mid_log] = 1
        out[y_log < low_log] = 0
        return out.reshape(-1)
    else:
        out = torch.full_like(y_log, fill_value=2, dtype=torch.long)
        out = torch.where(y_log < mid_log, torch.ones_like(out), out)
        out = torch.where(y_log < low_log, torch.zeros_like(out), out)
        return out.view(-1)


# =========================
# Stable Multi-objective Loss
# =========================
class StableMultiObjectiveBERLoss(nn.Module):
    """
    Stable multi-objective loss:
      - Huber loss in log10(BER) space
      - Huber loss in raw BER space
      - optional regime weighting based on raw BER

    Critical stability fix:
      pred_log is CLAMPED BEFORE conversion to raw BER to avoid overflow in 10**pred_log.
    """
    def __init__(
        self,
        log_delta=0.5,
        raw_delta=0.01,
        alpha_log=0.9,
        beta_raw=0.1,
        use_regime_weights=False,
        low_thr=1e-6,
        mid_thr=1e-3,
        w_low=1.0,
        w_mid=1.25,
        w_high=1.75,
        min_log_ber=-12.0,
        max_log_ber=0.0,
    ):
        super().__init__()
        self.log_delta = log_delta
        self.raw_delta = raw_delta
        self.alpha_log = alpha_log
        self.beta_raw = beta_raw

        self.use_regime_weights = use_regime_weights
        self.low_thr = low_thr
        self.mid_thr = mid_thr
        self.w_low = w_low
        self.w_mid = w_mid
        self.w_high = w_high

        self.min_log_ber = min_log_ber
        self.max_log_ber = max_log_ber

    @staticmethod
    def huber_elementwise(pred, target, delta):
        err = pred - target
        abs_err = err.abs()
        return torch.where(
            abs_err < delta,
            0.5 * err * err,
            delta * (abs_err - 0.5 * delta)
        )

    def forward(self, pred_log, target_log):
        log_loss = self.huber_elementwise(pred_log, target_log, self.log_delta)

        pred_log_for_raw = pred_log.clamp(min=self.min_log_ber, max=self.max_log_ber)
        target_log_for_raw = target_log.clamp(min=self.min_log_ber, max=self.max_log_ber)

        pred_raw = torch.pow(10.0, pred_log_for_raw)
        target_raw = torch.pow(10.0, target_log_for_raw)

        raw_loss = self.huber_elementwise(pred_raw, target_raw, self.raw_delta)

        total = self.alpha_log * log_loss + self.beta_raw * raw_loss

        if self.use_regime_weights:
            weights = torch.full_like(target_raw, self.w_high)
            weights = torch.where(
                target_raw < self.mid_thr,
                torch.full_like(weights, self.w_mid),
                weights
            )
            weights = torch.where(
                target_raw < self.low_thr,
                torch.full_like(weights, self.w_low),
                weights
            )
            total = total * weights

        return total.mean()


# =========================
# MoE Head + Aux Losses
# =========================
class MoERegressionHead(nn.Module):
    """
    Soft mixture-of-experts head.

    Output:
      pred_log: [B, 1]
      gate_probs: [B, K]
      expert_preds: [B, K]
    """
    def __init__(self, in_dim, num_experts=3, gate_hidden=128, expert_hidden=(512, 256, 128), dropout=0.15):
        super().__init__()
        self.num_experts = num_experts

        self.gate = nn.Sequential(
            nn.Linear(in_dim, gate_hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(gate_hidden, num_experts)
        )

        experts = []
        for _ in range(num_experts):
            layers = []
            prev = in_dim
            for h in expert_hidden:
                layers.extend([
                    nn.Linear(prev, h),
                    nn.GELU(),
                    nn.Dropout(dropout),
                ])
                prev = h
            layers.append(nn.Linear(prev, 1))
            experts.append(nn.Sequential(*layers))
        self.experts = nn.ModuleList(experts)

    def forward(self, x):
        gate_logits = self.gate(x)                   # [B, K]
        gate_probs = torch.softmax(gate_logits, dim=1)

        expert_preds = torch.cat(
            [expert(x) for expert in self.experts], dim=1
        )                                            # [B, K]

        pred = torch.sum(gate_probs * expert_preds, dim=1, keepdim=True)
        return pred, gate_probs, expert_preds


def moe_balance_loss(gate_probs):
    """
    Encourage average routing to not collapse onto one expert.
    """
    num_experts = gate_probs.size(1)
    avg_usage = gate_probs.mean(dim=0)
    target = torch.full_like(avg_usage, 1.0 / num_experts)
    return torch.mean((avg_usage - target) ** 2)


# =========================
# Blocks
# =========================
class SqueezeExcite1D(nn.Module):
    def __init__(self, channels, reduction=8):
        super().__init__()
        hidden = max(channels // reduction, 8)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Sequential(
            nn.Conv1d(channels, hidden, kernel_size=1),
            nn.GELU(),
            nn.Conv1d(hidden, channels, kernel_size=1),
            nn.Sigmoid()
        )

    def forward(self, x):
        scale = self.pool(x)
        scale = self.fc(scale)
        return x * scale


class ConvBNAct(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size, dilation=1, stride=1, groups=1):
        super().__init__()
        padding = ((kernel_size - 1) // 2) * dilation
        self.block = nn.Sequential(
            nn.Conv1d(
                in_ch, out_ch,
                kernel_size=kernel_size,
                stride=stride,
                padding=padding,
                dilation=dilation,
                groups=groups,
                bias=False
            ),
            nn.BatchNorm1d(out_ch),
            nn.GELU()
        )

    def forward(self, x):
        return self.block(x)


class MultiScaleResidualBlock(nn.Module):
    def __init__(self, in_ch, out_ch, dropout=0.1, se=True):
        super().__init__()
        b1 = out_ch // 3
        b2 = out_ch // 3
        b3 = out_ch - b1 - b2

        self.branch1 = ConvBNAct(in_ch, b1, kernel_size=3, dilation=1)
        self.branch2 = ConvBNAct(in_ch, b2, kernel_size=5, dilation=1)
        self.branch3 = ConvBNAct(in_ch, b3, kernel_size=3, dilation=2)

        self.mix = nn.Sequential(
            nn.Conv1d(out_ch, out_ch, kernel_size=1, bias=False),
            nn.BatchNorm1d(out_ch),
            nn.GELU(),
            nn.Dropout(dropout)
        )

        self.se = SqueezeExcite1D(out_ch) if se else nn.Identity()

        self.skip = (
            nn.Sequential(
                nn.Conv1d(in_ch, out_ch, kernel_size=1, bias=False),
                nn.BatchNorm1d(out_ch)
            )
            if in_ch != out_ch else nn.Identity()
        )

        self.out_act = nn.GELU()

    def forward(self, x):
        residual = self.skip(x)
        out = torch.cat(
            [self.branch1(x), self.branch2(x), self.branch3(x)],
            dim=1
        )
        out = self.mix(out)
        out = self.se(out)
        out = out + residual
        out = self.out_act(out)
        return out


class DownsampleResidualBlock(nn.Module):
    def __init__(self, in_ch, out_ch, dropout=0.1, se=True):
        super().__init__()
        self.pre = MultiScaleResidualBlock(in_ch, out_ch, dropout=dropout, se=se)
        self.down = nn.Sequential(
            nn.Conv1d(
                out_ch, out_ch,
                kernel_size=3,
                stride=2,
                padding=1,
                bias=False
            ),
            nn.BatchNorm1d(out_ch),
            nn.GELU()
        )

    def forward(self, x):
        x = self.pre(x)
        x = self.down(x)
        return x


class ThresholdFiLM(nn.Module):
    def __init__(self, threshold_dim, feature_dim, hidden_dim=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(threshold_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, feature_dim * 2)
        )
        self.scale = nn.Parameter(torch.tensor(0.25, dtype=torch.float32))

    def forward(self, x, thr_emb):
        gamma_beta = self.net(thr_emb)
        gamma, beta = torch.chunk(gamma_beta, 2, dim=1)
        gamma = gamma.unsqueeze(-1)
        beta = beta.unsqueeze(-1)
        return x * (1.0 + self.scale * gamma) + self.scale * beta


class SequenceStem(nn.Module):
    def __init__(self, in_ch, stem_ch=32):
        super().__init__()
        self.net = nn.Sequential(
            ConvBNAct(in_ch, stem_ch, kernel_size=5),
            MultiScaleResidualBlock(stem_ch, stem_ch, dropout=0.05, se=True)
        )

    def forward(self, x):
        return self.net(x)


class LightSelfAttention1D(nn.Module):
    def __init__(self, dim, num_heads=4, mlp_ratio=2.0, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(
            embed_dim=dim,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True
        )
        self.norm2 = nn.LayerNorm(dim)

        hidden = int(dim * mlp_ratio)
        self.mlp = nn.Sequential(
            nn.Linear(dim, hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, dim),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        x_seq = x.transpose(1, 2)  # [B, L, C]
        y = self.norm1(x_seq)
        attn_out, _ = self.attn(y, y, y, need_weights=False)
        x_seq = x_seq + attn_out
        x_seq = x_seq + self.mlp(self.norm2(x_seq))
        return x_seq.transpose(1, 2)


class MultiHeadAttentionPooling(nn.Module):
    def __init__(self, input_dim, attn_dim=128, num_heads=4):
        super().__init__()
        self.score = nn.Sequential(
            nn.Conv1d(input_dim, attn_dim, kernel_size=1),
            nn.GELU(),
            nn.Conv1d(attn_dim, num_heads, kernel_size=1)
        )
        self.num_heads = num_heads
        self.input_dim = input_dim

    def forward(self, x):
        logits = self.score(x)
        weights = torch.softmax(logits, dim=-1)
        pooled = torch.einsum("bhl,bcl->bhc", weights, x)
        pooled = pooled.reshape(x.size(0), self.num_heads * self.input_dim)
        return pooled


class StagePooling(nn.Module):
    def __init__(self, channels, num_attn_heads=2, attn_dim=64):
        super().__init__()
        self.attn_pool = MultiHeadAttentionPooling(
            input_dim=channels,
            attn_dim=attn_dim,
            num_heads=num_attn_heads
        )
        self.out_dim = num_attn_heads * channels + 3 * channels

    def forward(self, x):
        attn = self.attn_pool(x)
        mean = x.mean(dim=-1)
        std = x.std(dim=-1, unbiased=False)
        maxv = x.amax(dim=-1)
        return torch.cat([attn, mean, std, maxv], dim=1)


# =========================
# Model
# =========================
class BERMultiScaleResCNNMoE(nn.Module):
    """
    Input seq channels:
      0 -> scaled tap means multiplied by number of molecules
      1 -> tap variances
      2 -> normalized position
      3 -> abs(scaled tap means)
      4 -> raw-ish log SNR proxy (scaled separately)
      5 -> first-position indicator
      6 -> z0 decision margin feature (repeated across sequence)
      7 -> z1 decision margin feature (repeated across sequence)

    Additional learned absolute positional embedding is injected in the model.

    Output:
      predicted log10(BER)
    """
    def __init__(
        self,
        seq_len,
        threshold_dim=1,
        stem_ch=32,
        channels=(64, 96, 128),
        dropout=0.10,
        stage_pool_heads=(2, 2, 2, 4),
        attn_heads=4,
        pos_emb_dim=8,
        use_self_attention=True,
        num_experts=3,
    ):
        super().__init__()

        self.seq_len = seq_len
        self.pos_emb_dim = pos_emb_dim
        self.use_self_attention = use_self_attention
        self.num_experts = num_experts

        self.mean_stem = SequenceStem(in_ch=1, stem_ch=stem_ch)
        self.var_stem = SequenceStem(in_ch=1, stem_ch=stem_ch)

        # extra_x contains:
        # [normalized position, abs tap, snr proxy, first flag, z0, z1]
        self.extra_proj = nn.Sequential(
            nn.Conv1d(6, stem_ch, kernel_size=1, bias=False),
            nn.BatchNorm1d(stem_ch),
            nn.GELU()
        )

        # learned absolute positional embedding: [1, pos_emb_dim, L]
        self.pos_embedding = nn.Parameter(
            torch.randn(1, pos_emb_dim, seq_len) * 0.02
        )

        self.pos_emb_proj = nn.Sequential(
            nn.Conv1d(pos_emb_dim, stem_ch, kernel_size=1, bias=False),
            nn.BatchNorm1d(stem_ch),
            nn.GELU()
        )

        fusion_in = stem_ch * 4

        self.thr_embed = nn.Sequential(
            nn.Linear(threshold_dim, 32),
            nn.GELU(),
            nn.Linear(32, 32),
            nn.GELU()
        )

        self.stage1 = MultiScaleResidualBlock(fusion_in, channels[0], dropout=dropout, se=True)
        self.film1 = ThresholdFiLM(32, channels[0], hidden_dim=64)

        self.stage2 = DownsampleResidualBlock(channels[0], channels[1], dropout=dropout, se=True)
        self.film2 = ThresholdFiLM(32, channels[1], hidden_dim=64)

        self.stage3 = DownsampleResidualBlock(channels[1], channels[2], dropout=dropout, se=True)
        self.film3 = ThresholdFiLM(32, channels[2], hidden_dim=64)

        if self.use_self_attention:
            self.self_attn = LightSelfAttention1D(
                dim=channels[2],
                num_heads=attn_heads,
                mlp_ratio=2.0,
                dropout=dropout
            )
        else:
            self.self_attn = nn.Identity()

        self.bottleneck = nn.Sequential(
            MultiScaleResidualBlock(channels[2], channels[2], dropout=dropout, se=True),
            MultiScaleResidualBlock(channels[2], channels[2], dropout=dropout, se=True),
        )

        self.pool1 = StagePooling(channels[0], num_attn_heads=stage_pool_heads[0], attn_dim=64)
        self.pool2 = StagePooling(channels[1], num_attn_heads=stage_pool_heads[1], attn_dim=64)
        self.pool3 = StagePooling(channels[2], num_attn_heads=stage_pool_heads[2], attn_dim=64)
        self.poolb = StagePooling(channels[2], num_attn_heads=stage_pool_heads[3], attn_dim=128)

        head_in = (
            self.pool1.out_dim +
            self.pool2.out_dim +
            self.pool3.out_dim +
            self.poolb.out_dim +
            32
        )

        self.moe_head = MoERegressionHead(
            in_dim=head_in,
            num_experts=num_experts,
            gate_hidden=128,
            expert_hidden=(512, 256, 128),
            dropout=0.15
        )

    def extract_fused_features(self, seq, threshold):
        mean_x = seq[:, 0:1, :]
        var_x = seq[:, 1:2, :]
        extra_x = seq[:, 2:8, :]  # pos, abs, snr, first_flag, z0, z1

        mean_f = self.mean_stem(mean_x)
        var_f = self.var_stem(var_x)
        extra_f = self.extra_proj(extra_x)

        pos_emb = self.pos_embedding.expand(seq.size(0), -1, -1)
        pos_f = self.pos_emb_proj(pos_emb)

        x = torch.cat([mean_f, var_f, extra_f, pos_f], dim=1)
        thr = self.thr_embed(threshold)

        x1 = self.stage1(x)
        x1 = self.film1(x1, thr)

        x2 = self.stage2(x1)
        x2 = self.film2(x2, thr)

        x3 = self.stage3(x2)
        x3 = self.film3(x3, thr)

        x3 = self.self_attn(x3)
        xb = self.bottleneck(x3)

        p1 = self.pool1(x1)
        p2 = self.pool2(x2)
        p3 = self.pool3(x3)
        pb = self.poolb(xb)

        fused = torch.cat([p1, p2, p3, pb, thr], dim=1)
        return fused

    def forward(self, seq, threshold, return_aux=False):
        fused = self.extract_fused_features(seq, threshold)
        pred_log, gate_probs, expert_preds = self.moe_head(fused)

        if return_aux:
            return pred_log, gate_probs, expert_preds
        return pred_log


# =========================
# Data
# =========================
def prepare_data(csv_path, batch_size=256, nrows=5000000, num_workers=0):
    df = pd.read_csv(csv_path, nrows=nrows)

    if "mem_len" not in df.columns:
        raise ValueError("Required column 'mem_len' not found.")
    if "N" not in df.columns:
        raise ValueError("Required column 'N' not found.")

    df = df[df["mem_len"] != 1].copy()

    tap_cols = get_sorted_seq_cols(df.columns, "tap")
    var_cols = get_sorted_seq_cols(df.columns, "var")

    if not tap_cols:
        raise ValueError("No tap_* columns found.")
    if not var_cols:
        raise ValueError("No var_* columns found.")
    if len(tap_cols) != len(var_cols):
        raise ValueError(
            f"tap/var length mismatch: {len(tap_cols)} tap cols vs {len(var_cols)} var cols"
        )

    required_cols = tap_cols + var_cols + ["threshold", "BER", "N"]
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    df[tap_cols] = df[tap_cols].fillna(0.0)
    df[var_cols] = df[var_cols].fillna(0.0)
    df["threshold"] = df["threshold"].fillna(0.0)
    df["BER"] = df["BER"].fillna(0.0)
    df["N"] = df["N"].fillna(0.0)

    df = df[(df["threshold"] > 0) & (df["BER"] > 0) & (df["N"] > 0)].copy()

    X_taps_raw = df[tap_cols].to_numpy(dtype=np.float32)
    X_vars_raw = df[var_cols].to_numpy(dtype=np.float32)
    num_molecules = df["N"].to_numpy(dtype=np.float32).reshape(-1, 1)

    X_thr_raw = df["threshold"].to_numpy(dtype=np.float32).reshape(-1, 1)
    y_raw = df["BER"].to_numpy(dtype=np.float32).reshape(-1, 1)

    X_thr = np.log10(X_thr_raw + EPS).astype(np.float32)
    y_log = np.log10(y_raw + EPS).astype(np.float32)

    # Mean is scaled to expected count-like value
    X_taps_feat = (X_taps_raw * num_molecules).astype(np.float32)

    if np.any(X_vars_raw < 0):
        raise ValueError("Variance columns contain negative values; cannot use them safely.")

    # Use variance directly (not std)
    X_vars_feat = X_vars_raw.astype(np.float32)

    abs_taps_raw = np.abs(X_taps_feat).astype(np.float32)

    # SNR proxy based on count-scaled mean and variance
    snr_raw = np.log10((X_taps_feat ** 2) / (X_vars_raw + EPS) + EPS).astype(np.float32)

    # -------------------------
    # Global decision features
    # -------------------------
    first_mean = X_taps_feat[:, 0:1]
    past_means = X_taps_feat[:, 1:]
    first_var = X_vars_feat[:, 0:1]
    past_vars = X_vars_feat[:, 1:]

    # Expected interference from past bits with P(bit=1)=0.5
    mu0_raw = 0.5 * np.sum(past_means, axis=1, keepdims=True).astype(np.float32)
    mu1_raw = (first_mean + mu0_raw).astype(np.float32)

    # Practical first-order approximation for aggregate conditional variances
    var0_raw = (0.5 * np.sum(past_vars, axis=1, keepdims=True)).astype(np.float32)
    var1_raw = (first_var + var0_raw).astype(np.float32)

    std0_raw = np.sqrt(np.maximum(var0_raw, EPS)).astype(np.float32)
    std1_raw = np.sqrt(np.maximum(var1_raw, EPS)).astype(np.float32)

    z0_raw = ((X_thr_raw - mu0_raw) / (std0_raw + EPS)).astype(np.float32)
    z1_raw = ((mu1_raw - X_thr_raw) / (std1_raw + EPS)).astype(np.float32)

    strat_labels = make_strat_bins(y_log, n_bins=10)

    split_args = dict(test_size=0.30, random_state=42)
    if strat_labels is not None:
        split_args["stratify"] = strat_labels

    (
        t_taps_raw, temp_taps_raw,
        t_vars_raw, temp_vars_raw,
        t_abs_raw, temp_abs_raw,
        t_snr_raw, temp_snr_raw,
        t_z0_raw, temp_z0_raw,
        t_z1_raw, temp_z1_raw,
        t_thr, temp_thr,
        t_y_log, temp_y_log
    ) = train_test_split(
        X_taps_feat, X_vars_feat, abs_taps_raw, snr_raw, z0_raw, z1_raw, X_thr, y_log,
        **split_args
    )

    temp_strat = make_strat_bins(temp_y_log, n_bins=6)

    split_args2 = dict(test_size=0.50, random_state=42)
    if temp_strat is not None:
        split_args2["stratify"] = temp_strat

    (
        v_taps_raw, te_taps_raw,
        v_vars_raw, te_vars_raw,
        v_abs_raw, te_abs_raw,
        v_snr_raw, te_snr_raw,
        v_z0_raw, te_z0_raw,
        v_z1_raw, te_z1_raw,
        v_thr, te_thr,
        v_y_log, te_y_log
    ) = train_test_split(
        temp_taps_raw, temp_vars_raw, temp_abs_raw, temp_snr_raw,
        temp_z0_raw, temp_z1_raw, temp_thr, temp_y_log,
        **split_args2
    )

    tap_scaler = StandardScaler()
    var_scaler = StandardScaler()
    abs_scaler = StandardScaler()
    snr_scaler = StandardScaler()
    z0_scaler = StandardScaler()
    z1_scaler = StandardScaler()
    thr_scaler = StandardScaler()

    t_taps = tap_scaler.fit_transform(t_taps_raw).astype(np.float32)
    v_taps = tap_scaler.transform(v_taps_raw).astype(np.float32)
    te_taps = tap_scaler.transform(te_taps_raw).astype(np.float32)

    t_vars = var_scaler.fit_transform(t_vars_raw).astype(np.float32)
    v_vars = var_scaler.transform(v_vars_raw).astype(np.float32)
    te_vars = var_scaler.transform(te_vars_raw).astype(np.float32)

    t_abs = abs_scaler.fit_transform(t_abs_raw).astype(np.float32)
    v_abs = abs_scaler.transform(v_abs_raw).astype(np.float32)
    te_abs = abs_scaler.transform(te_abs_raw).astype(np.float32)

    t_snr = snr_scaler.fit_transform(t_snr_raw).astype(np.float32)
    v_snr = snr_scaler.transform(v_snr_raw).astype(np.float32)
    te_snr = snr_scaler.transform(te_snr_raw).astype(np.float32)

    t_z0 = z0_scaler.fit_transform(t_z0_raw).astype(np.float32)
    v_z0 = z0_scaler.transform(v_z0_raw).astype(np.float32)
    te_z0 = z0_scaler.transform(te_z0_raw).astype(np.float32)

    t_z1 = z1_scaler.fit_transform(t_z1_raw).astype(np.float32)
    v_z1 = z1_scaler.transform(v_z1_raw).astype(np.float32)
    te_z1 = z1_scaler.transform(te_z1_raw).astype(np.float32)

    t_thr = thr_scaler.fit_transform(t_thr).astype(np.float32)
    v_thr = thr_scaler.transform(v_thr).astype(np.float32)
    te_thr = thr_scaler.transform(te_thr).astype(np.float32)

    L = t_taps.shape[1]
    pos = np.linspace(0.0, 1.0, L, dtype=np.float32)

    def build_features(taps_scaled, vars_scaled, abs_scaled, snr_scaled, z0_scaled, z1_scaled):
        n = taps_scaled.shape[0]

        pos_ch = np.tile(pos, (n, 1)).astype(np.float32)

        first_flag = np.zeros((n, L), dtype=np.float32)
        first_flag[:, 0] = 1.0

        z0_ch = np.tile(z0_scaled, (1, L)).astype(np.float32)
        z1_ch = np.tile(z1_scaled, (1, L)).astype(np.float32)

        seq = np.stack(
            [
                taps_scaled,
                vars_scaled,
                pos_ch,
                abs_scaled,
                snr_scaled,
                first_flag,
                z0_ch,
                z1_ch,
            ],
            axis=1
        ).astype(np.float32)
        return seq

    t_seq = build_features(t_taps, t_vars, t_abs, t_snr, t_z0, t_z1)
    v_seq = build_features(v_taps, v_vars, v_abs, v_snr, v_z0, v_z1)
    te_seq = build_features(te_taps, te_vars, te_abs, te_snr, te_z0, te_z1)

    train_ds = TensorDataset(
        torch.from_numpy(t_seq),
        torch.from_numpy(t_thr),
        torch.from_numpy(t_y_log)
    )
    val_ds = TensorDataset(
        torch.from_numpy(v_seq),
        torch.from_numpy(v_thr),
        torch.from_numpy(v_y_log)
    )
    test_ds = TensorDataset(
        torch.from_numpy(te_seq),
        torch.from_numpy(te_thr),
        torch.from_numpy(te_y_log)
    )

    pin_mem = torch.cuda.is_available()

    train_loader = DataLoader(
        train_ds,
        batch_size=batch_size,
        shuffle=True,
        pin_memory=pin_mem,
        num_workers=num_workers
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=batch_size,
        shuffle=False,
        pin_memory=pin_mem,
        num_workers=num_workers
    )
    test_loader = DataLoader(
        test_ds,
        batch_size=batch_size,
        shuffle=False,
        pin_memory=pin_mem,
        num_workers=num_workers
    )

    scalers = {
        "tap_scaler": tap_scaler,
        "var_scaler": var_scaler,
        "abs_scaler": abs_scaler,
        "snr_scaler": snr_scaler,
        "z0_scaler": z0_scaler,
        "z1_scaler": z1_scaler,
        "thr_scaler": thr_scaler,
        "tap_cols": tap_cols,
        "var_cols": var_cols,
        "seq_channels": 8,
        "feature_order": [
            "tap_scaled_mean",
            "variance",
            "pos",
            "abs_tap_scaled_mean",
            "snr_proxy_log",
            "first_flag",
            "z0_margin",
            "z1_margin",
        ],
        "seq_len": L,
        "uses_learned_positional_embedding": True,
        "pos_emb_dim": 8,
        "uses_self_attention": True,
        "uses_moe_head": True,
        "num_experts": 3,
        "preprocessing": {
            "tap_transform": "tap_mean * num_molecules",
            "var_transform": "use raw variance",
            "z0_definition": "z0 = (threshold_raw - mu0) / sqrt(var0 + eps)",
            "z1_definition": "z1 = (mu1 - threshold_raw) / sqrt(var1 + eps)",
            "mu0_definition": "0.5 * sum(past_scaled_means)",
            "mu1_definition": "first_scaled_mean + mu0",
            "var0_definition": "0.5 * sum(past_variances)",
            "var1_definition": "first_variance + var0",
        },
    }

    return train_loader, val_loader, test_loader, scalers, len(tap_cols)


# =========================
# Evaluation
# =========================
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    all_preds_log = []
    all_targets_log = []
    all_gate_probs = []

    with torch.no_grad():
        for b_seq, b_thr, b_y_log in loader:
            b_seq = b_seq.to(device, non_blocking=True)
            b_thr = b_thr.to(device, non_blocking=True)
            b_y_log = b_y_log.to(device, non_blocking=True)

            pred_log, gate_probs, _ = model(b_seq, b_thr, return_aux=True)

            if has_nonfinite_tensor(pred_log):
                raise RuntimeError("Non-finite prediction detected during evaluation.")

            loss = criterion(pred_log, b_y_log)

            if has_nonfinite_tensor(loss):
                raise RuntimeError("Non-finite loss detected during evaluation.")

            total_loss += loss.item()
            all_preds_log.append(pred_log.cpu().numpy())
            all_targets_log.append(b_y_log.cpu().numpy())
            all_gate_probs.append(gate_probs.cpu().numpy())

    avg_loss = total_loss / max(len(loader), 1)

    preds_log = np.vstack(all_preds_log)
    targets_log = np.vstack(all_targets_log)
    gate_probs = np.vstack(all_gate_probs)

    preds_raw = np.clip(10 ** np.clip(preds_log, -12.0, 0.0), EPS, 1.0)
    targets_raw = np.clip(10 ** np.clip(targets_log, -12.0, 0.0), EPS, 1.0)

    rmse_log = float(np.sqrt(np.mean((preds_log - targets_log) ** 2)))
    mae_log = float(np.mean(np.abs(preds_log - targets_log)))
    factor_error = float(10 ** rmse_log)

    rmse_raw = float(np.sqrt(np.mean((preds_raw - targets_raw) ** 2)))
    mae_raw = float(np.mean(np.abs(preds_raw - targets_raw)))

    avg_gate_usage = gate_probs.mean(axis=0)

    return avg_loss, rmse_log, mae_log, factor_error, rmse_raw, mae_raw, avg_gate_usage


def evaluate_by_target_range(model, loader, device):
    model.eval()
    all_preds_log = []
    all_targets_log = []
    all_gate_probs = []

    with torch.no_grad():
        for b_seq, b_thr, b_y_log in loader:
            b_seq = b_seq.to(device, non_blocking=True)
            b_thr = b_thr.to(device, non_blocking=True)
            pred_log, gate_probs, _ = model(b_seq, b_thr, return_aux=True)

            if has_nonfinite_tensor(pred_log):
                raise RuntimeError("Non-finite prediction detected during per-range evaluation.")

            all_preds_log.append(pred_log.cpu().numpy())
            all_targets_log.append(b_y_log.numpy())
            all_gate_probs.append(gate_probs.cpu().numpy())

    preds_log = np.vstack(all_preds_log).reshape(-1)
    targets_log = np.vstack(all_targets_log).reshape(-1)
    gate_probs = np.vstack(all_gate_probs)

    preds_raw = np.clip(10 ** np.clip(preds_log, -12.0, 0.0), EPS, 1.0)
    targets_raw = np.clip(10 ** np.clip(targets_log, -12.0, 0.0), EPS, 1.0)

    ranges = {
        "low_BER(y<1e-6)": targets_raw < 1e-6,
        "mid_BER(1e-6<=y<1e-3)": (targets_raw >= 1e-6) & (targets_raw < 1e-3),
        "high_BER(y>=1e-3)": targets_raw >= 1e-3,
    }

    metrics = {}
    for name, mask in ranges.items():
        if np.any(mask):
            rmse_log = float(np.sqrt(np.mean((preds_log[mask] - targets_log[mask]) ** 2)))
            mae_log = float(np.mean(np.abs(preds_log[mask] - targets_log[mask])))
            rmse_raw = float(np.sqrt(np.mean((preds_raw[mask] - targets_raw[mask]) ** 2)))
            mae_raw = float(np.mean(np.abs(preds_raw[mask] - targets_raw[mask])))
            gate_usage = gate_probs[mask].mean(axis=0)

            metrics[name] = {
                "count": int(mask.sum()),
                "rmse_log": rmse_log,
                "mae_log": mae_log,
                "factor_error": float(10 ** rmse_log),
                "rmse_raw": rmse_raw,
                "mae_raw": mae_raw,
                "avg_gate_usage": gate_usage.tolist(),
            }
        else:
            metrics[name] = None

    return metrics


# =========================
# Training
# =========================
def train_engine():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Executing on: {device}")

    train_loader, val_loader, test_loader, scalers, seq_len = prepare_data(
        DATA_PATH,
        batch_size=256,
        nrows=500000,
        num_workers=0
    )

    joblib.dump(scalers, SCALER_SAVE_PATH)

    model = BERMultiScaleResCNNMoE(
        seq_len=seq_len,
        threshold_dim=1,
        stem_ch=32,
        channels=(64, 96, 128),
        dropout=0.10,
        stage_pool_heads=(2, 2, 2, 4),
        attn_heads=4,
        pos_emb_dim=8,
        use_self_attention=True,
        num_experts=3,
    ).to(device)

    optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)

    criterion = StableMultiObjectiveBERLoss(
        log_delta=0.5,
        raw_delta=0.01,
        alpha_log=0.9,
        beta_raw=0.1,
        use_regime_weights=False,
        low_thr=1e-6,
        mid_thr=1e-3,
        w_low=1.0,
        w_mid=1.25,
        w_high=1.75,
        min_log_ber=-12.0,
        max_log_ber=0.0,
    )

    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        patience=4,
        factor=0.5
    )

    # Auxiliary MoE loss weights
    lambda_gate_ce = 0.05
    lambda_balance = 0.01

    gate_ce_loss_fn = nn.NLLLoss()

    best_val_loss = float("inf")
    best_state = None
    patience = 12
    wait = 0
    min_epochs_before_early_stop = 60

    print(f"Detected sequence length L = {seq_len}")

    training_broke = False

    for epoch in range(100):
        model.train()

        running_main_loss = 0.0
        running_total_loss = 0.0
        running_gate_ce = 0.0
        running_balance = 0.0
        running_abs_err_log = 0.0
        running_sq_err_log = 0.0
        running_abs_err_raw = 0.0
        running_count = 0
        running_gate_usage_sum = torch.zeros(model.num_experts, dtype=torch.float64)

        for batch_idx, (b_seq, b_thr, b_y_log) in enumerate(train_loader):
            b_seq = b_seq.to(device, non_blocking=True)
            b_thr = b_thr.to(device, non_blocking=True)
            b_y_log = b_y_log.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            pred_log, gate_probs, _ = model(b_seq, b_thr, return_aux=True)

            if has_nonfinite_tensor(pred_log):
                print(f"Non-finite prediction detected at epoch {epoch+1}, batch {batch_idx+1}.")
                training_broke = True
                break

            main_loss = criterion(pred_log, b_y_log)
            if has_nonfinite_tensor(main_loss):
                print(f"Non-finite main loss detected at epoch {epoch+1}, batch {batch_idx+1}.")
                training_broke = True
                break

            regime_targets = log10ber_to_regime_index(b_y_log)
            gate_log_probs = torch.log(gate_probs.clamp_min(1e-8))
            gate_ce = gate_ce_loss_fn(gate_log_probs, regime_targets)

            balance = moe_balance_loss(gate_probs)

            loss = main_loss + lambda_gate_ce * gate_ce + lambda_balance * balance

            if has_nonfinite_tensor(loss):
                print(f"Non-finite total loss detected at epoch {epoch+1}, batch {batch_idx+1}.")
                training_broke = True
                break

            loss.backward()

            grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            if not torch.isfinite(grad_norm):
                print(f"Non-finite gradient norm detected at epoch {epoch+1}, batch {batch_idx+1}.")
                training_broke = True
                break

            optimizer.step()

            bad_param = False
            for name, param in model.named_parameters():
                if param.requires_grad and param.data is not None and not torch.isfinite(param.data).all():
                    print(f"Non-finite parameter detected after optimizer step: {name}")
                    bad_param = True
                    break
            if bad_param:
                training_broke = True
                break

            # Running training stats without full train-loader evaluation
            batch_size_actual = b_seq.size(0)
            running_main_loss += main_loss.item() * batch_size_actual
            running_total_loss += loss.item() * batch_size_actual
            running_gate_ce += gate_ce.item() * batch_size_actual
            running_balance += balance.item() * batch_size_actual

            pred_log_det = pred_log.detach()
            target_log_det = b_y_log.detach()

            abs_err_log = torch.abs(pred_log_det - target_log_det)
            sq_err_log = (pred_log_det - target_log_det) ** 2

            pred_raw = torch.pow(10.0, pred_log_det.clamp(-12.0, 0.0))
            target_raw = torch.pow(10.0, target_log_det.clamp(-12.0, 0.0))
            abs_err_raw = torch.abs(pred_raw - target_raw)

            running_abs_err_log += abs_err_log.sum().item()
            running_sq_err_log += sq_err_log.sum().item()
            running_abs_err_raw += abs_err_raw.sum().item()
            running_count += batch_size_actual

            running_gate_usage_sum += gate_probs.detach().sum(dim=0).cpu().double()

        if training_broke:
            print("Training stopped because non-finite values were detected.")
            break

        if running_count == 0:
            print("Training stopped because no valid batches were processed.")
            break

        train_main_loss = running_main_loss / running_count
        train_total_loss = running_total_loss / running_count
        train_gate_ce = running_gate_ce / running_count
        train_balance = running_balance / running_count
        train_mae_log = running_abs_err_log / running_count
        train_rmse_log = float(np.sqrt(running_sq_err_log / running_count))
        train_factor = float(10 ** train_rmse_log)
        train_mae_raw = running_abs_err_raw / running_count
        train_gate_usage = (running_gate_usage_sum / running_count).numpy()

        try:
            val_loss, val_rmse_log, val_mae_log, val_factor, val_rmse_raw, val_mae_raw, val_gate_usage = evaluate(
                model, val_loader, criterion, device
            )
        except RuntimeError as e:
            print(f"Evaluation failed at epoch {epoch+1}: {e}")
            break

        scheduler.step(val_loss)
        current_lr = optimizer.param_groups[0]["lr"]

        print(
            f"Epoch {epoch+1:03d} | "
            f"LR: {current_lr:.2e} | "
            f"Train Main Loss: {train_main_loss:.4f} | "
            f"Train Total Loss: {train_total_loss:.4f} | "
            f"Val Loss: {val_loss:.4f} | "
            f"Train RMSE(log10): {train_rmse_log:.4f} (~{train_factor:.2f}x) | "
            f"Val RMSE(log10): {val_rmse_log:.4f} (~{val_factor:.2f}x) | "
            f"Train MAE(log10): {train_mae_log:.4f} | "
            f"Val MAE(log10): {val_mae_log:.4f} | "
            f"Train MAE(raw): {train_mae_raw:.6f} | "
            f"Val MAE(raw): {val_mae_raw:.6f} | "
            f"Gate CE: {train_gate_ce:.4f} | "
            f"Balance: {train_balance:.6f}"
        )
        print(
            f"  Train gate usage: {np.round(train_gate_usage, 4).tolist()} | "
            f"Val gate usage: {np.round(val_gate_usage, 4).tolist()}"
        )

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())
            wait = 0
            print(f"  -> New best model saved at epoch {epoch+1} with Val Loss: {val_loss:.6f}")
        else:
            if epoch + 1 >= min_epochs_before_early_stop:
                wait += 1
                if wait >= patience:
                    print("Early stopping triggered.")
                    break

    torch.save(model.state_dict(), LAST_MODEL_SAVE_PATH)
    print(f"Last model saved to: {LAST_MODEL_SAVE_PATH}")

    if best_state is not None:
        model.load_state_dict(best_state)
        torch.save(model.state_dict(), BEST_MODEL_SAVE_PATH)
        print(f"Best model saved to: {BEST_MODEL_SAVE_PATH}")
    else:
        print("Warning: no valid best checkpoint was found.")

    test_loss, test_rmse_log, test_mae_log, test_factor, test_rmse_raw, test_mae_raw, test_gate_usage = evaluate(
        model, test_loader, criterion, device
    )

    print(
        f"Test Loss: {test_loss:.4f} | "
        f"Test RMSE(log10): {test_rmse_log:.4f} | "
        f"Test MAE(log10): {test_mae_log:.4f} | "
        f"Typical multiplicative error: ~{test_factor:.2f}x | "
        f"Test RMSE(raw): {test_rmse_raw:.6f} | "
        f"Test MAE(raw): {test_mae_raw:.6f}"
    )
    print(f"Test gate usage: {np.round(test_gate_usage, 4).tolist()}")

    range_metrics = evaluate_by_target_range(model, test_loader, device)
    print("\nPer-range test diagnostics:")
    for name, stats in range_metrics.items():
        if stats is None:
            print(f"{name}: no samples")
        else:
            print(
                f"{name} | count={stats['count']} | "
                f"RMSE(log10)={stats['rmse_log']:.4f} | "
                f"MAE(log10)={stats['mae_log']:.4f} | "
                f"factor~{stats['factor_error']:.2f}x | "
                f"RMSE(raw)={stats['rmse_raw']:.6f} | "
                f"MAE(raw)={stats['mae_raw']:.6f} | "
                f"gate_usage={np.round(stats['avg_gate_usage'], 4).tolist()}"
            )

    return model


if __name__ == "__main__":
    os.makedirs(BASE_DIR, exist_ok=True)

    if os.path.exists(DATA_PATH):
        print(f"Reading data from: {DATA_PATH}")
        trained_model = train_engine()
        print(f"Scalers saved to: {SCALER_SAVE_PATH}")
    else:
        print(f"Critical Error: Data file not found at {DATA_PATH}")

Reading data from: ./data_physics_with_variances_total.csv
Executing on: cuda
Detected sequence length L = 14
Epoch 001 | LR: 1.00e-04 | Train Main Loss: 0.3042 | Train Total Loss: 0.3163 | Val Loss: 0.1409 | Train RMSE(log10): 1.9911 (~97.98x) | Val RMSE(log10): 1.1409 (~13.83x) | Train MAE(log10): 0.8139 | Val MAE(log10): 0.4294 | Train MAE(raw): 0.060053 | Val MAE(raw): 0.032051 | Gate CE: 0.2215 | Balance: 0.100952
  Train gate usage: [0.1971, 0.0321, 0.7708] | Val gate usage: [0.21729999780654907, 0.029999999329447746, 0.7526999711990356]
  -> New best model saved at epoch 1 with Val Loss: 0.140922
Epoch 002 | LR: 1.00e-04 | Train Main Loss: 0.2023 | Train Total Loss: 0.2113 | Val Loss: 0.0896 | Train RMSE(log10): 1.4669 (~29.31x) | Val RMSE(log10): 0.8221 (~6.64x) | Train MAE(log10): 0.5705 | Val MAE(log10): 0.3011 | Train MAE(raw): 0.039366 | Val MAE(raw): 0.021893 | Gate CE: 0.1589 | Balance: 0.102969
  Train gate usage: [0.1908, 0.0319, 0.7773] | Val gate usage: [0.20479999482

Epoch 019 | LR: 1.00e-04 | Train Main Loss: 0.0686 | Train Total Loss: 0.0749 | Val Loss: 0.0282 | Train RMSE(log10): 0.6825 (~4.81x) | Val RMSE(log10): 0.4280 (~2.68x) | Train MAE(log10): 0.2387 | Val MAE(log10): 0.1179 | Train MAE(raw): 0.019141 | Val MAE(raw): 0.010968 | Gate CE: 0.1063 | Balance: 0.103571
  Train gate usage: [0.1573, 0.0585, 0.7842] | Val gate usage: [0.16220000386238098, 0.06019999831914902, 0.7775999903678894]
  -> New best model saved at epoch 19 with Val Loss: 0.028236
Epoch 020 | LR: 1.00e-04 | Train Main Loss: 0.0705 | Train Total Loss: 0.0768 | Val Loss: 0.0481 | Train RMSE(log10): 0.6938 (~4.94x) | Val RMSE(log10): 0.5313 (~3.40x) | Train MAE(log10): 0.2437 | Val MAE(log10): 0.1726 | Train MAE(raw): 0.019375 | Val MAE(raw): 0.013374 | Gate CE: 0.1060 | Balance: 0.103596
  Train gate usage: [0.1594, 0.0567, 0.7839] | Val gate usage: [0.17720000445842743, 0.05790000036358833, 0.7649999856948853]
Epoch 021 | LR: 1.00e-04 | Train Main Loss: 0.0670 | Train Total

Epoch 038 | LR: 2.50e-05 | Train Main Loss: 0.0365 | Train Total Loss: 0.0416 | Val Loss: 0.0216 | Train RMSE(log10): 0.4014 (~2.52x) | Val RMSE(log10): 0.3557 (~2.27x) | Train MAE(log10): 0.1563 | Val MAE(log10): 0.0917 | Train MAE(raw): 0.014548 | Val MAE(raw): 0.008324 | Gate CE: 0.0797 | Balance: 0.103670
  Train gate usage: [0.158, 0.0577, 0.7843] | Val gate usage: [0.1632000058889389, 0.06019999831914902, 0.7766000032424927]
Epoch 039 | LR: 2.50e-05 | Train Main Loss: 0.0362 | Train Total Loss: 0.0412 | Val Loss: 0.0242 | Train RMSE(log10): 0.4037 (~2.53x) | Val RMSE(log10): 0.3376 (~2.18x) | Train MAE(log10): 0.1550 | Val MAE(log10): 0.1005 | Train MAE(raw): 0.014526 | Val MAE(raw): 0.008837 | Gate CE: 0.0799 | Balance: 0.103648
  Train gate usage: [0.158, 0.0578, 0.7842] | Val gate usage: [0.16220000386238098, 0.06120000034570694, 0.7766000032424927]
Epoch 040 | LR: 2.50e-05 | Train Main Loss: 0.0360 | Train Total Loss: 0.0410 | Val Loss: 0.0230 | Train RMSE(log10): 0.3984 (~2.

Epoch 057 | LR: 6.25e-06 | Train Main Loss: 0.0283 | Train Total Loss: 0.0329 | Val Loss: 0.0274 | Train RMSE(log10): 0.3252 (~2.11x) | Val RMSE(log10): 0.3429 (~2.20x) | Train MAE(log10): 0.1342 | Val MAE(log10): 0.1104 | Train MAE(raw): 0.013485 | Val MAE(raw): 0.008937 | Gate CE: 0.0721 | Balance: 0.103449
  Train gate usage: [0.1581, 0.0582, 0.7837] | Val gate usage: [0.16539999842643738, 0.060499999672174454, 0.7741000056266785]
Epoch 058 | LR: 6.25e-06 | Train Main Loss: 0.0285 | Train Total Loss: 0.0332 | Val Loss: 0.0309 | Train RMSE(log10): 0.3300 (~2.14x) | Val RMSE(log10): 0.4066 (~2.55x) | Train MAE(log10): 0.1346 | Val MAE(log10): 0.1189 | Train MAE(raw): 0.013412 | Val MAE(raw): 0.009916 | Gate CE: 0.0717 | Balance: 0.103344
  Train gate usage: [0.1588, 0.0578, 0.7834] | Val gate usage: [0.16609999537467957, 0.06080000102519989, 0.7731999754905701]
Epoch 059 | LR: 6.25e-06 | Train Main Loss: 0.0287 | Train Total Loss: 0.0333 | Val Loss: 0.0164 | Train RMSE(log10): 0.3336 

In [2]:
import os
import re
import copy
import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torch.multiprocessing as mp
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

mp.set_sharing_strategy("file_system")

# =========================
# Configuration
# =========================
# BASE_DIR = "/mnt/erencem-ozbey/ber_estimation"
BASE_DIR = "./"
DATA_PATH = os.path.join(BASE_DIR, "data_physics_with_variances_total.csv")

BEST_MODEL_SAVE_PATH = os.path.join(
    BASE_DIR, "ber_multiscale_rescnn_moe_best_stable_large_scaled_means_var_z01_5M_real.pth"
)
LAST_MODEL_SAVE_PATH = os.path.join(
    BASE_DIR, "ber_multiscale_rescnn_moe_last_stable_large_scaled_means_var_z01_5M_real.pth"
)
SCALER_SAVE_PATH = os.path.join(
    BASE_DIR, "ber_multiscale_rescnn_moe_scalers_large_scaled_means_var_z01_5M_real.pkl"
)

EPS = 1e-12


# =========================
# Utilities
# =========================
def get_sorted_seq_cols(columns, prefix):
    pattern = re.compile(rf"^{re.escape(prefix)}_(\d+)$")
    matched = []
    for col in columns:
        m = pattern.match(col)
        if m:
            matched.append((int(m.group(1)), col))
    matched.sort(key=lambda x: x[0])
    return [col for _, col in matched]


def make_strat_bins(y_log, n_bins=10):
    """
    Build stratification bins from log10(BER).
    Falls back safely if quantile edges collapse.
    """
    y_flat = y_log.reshape(-1)
    quantiles = np.linspace(0, 1, n_bins + 1)
    edges = np.quantile(y_flat, quantiles)
    edges = np.unique(edges)

    if len(edges) < 3:
        return None

    bins = np.digitize(y_flat, edges[1:-1], right=True)
    counts = np.bincount(bins)
    if np.any(counts < 2):
        return None
    return bins


def has_nonfinite_tensor(x):
    return not torch.isfinite(x).all().item()


def log10ber_to_regime_index(y_log, low_thr=1e-6, mid_thr=1e-3):
    """
    Maps target log10(BER) to:
      0 -> low BER: y < 1e-6
      1 -> mid BER: 1e-6 <= y < 1e-3
      2 -> high BER: y >= 1e-3
    """
    low_log = np.log10(low_thr)
    mid_log = np.log10(mid_thr)

    if isinstance(y_log, np.ndarray):
        out = np.full_like(y_log, fill_value=2, dtype=np.int64)
        out[y_log < mid_log] = 1
        out[y_log < low_log] = 0
        return out.reshape(-1)
    else:
        out = torch.full_like(y_log, fill_value=2, dtype=torch.long)
        out = torch.where(y_log < mid_log, torch.ones_like(out), out)
        out = torch.where(y_log < low_log, torch.zeros_like(out), out)
        return out.view(-1)


# =========================
# Stable Multi-objective Loss
# =========================
class StableMultiObjectiveBERLoss(nn.Module):
    """
    Stable multi-objective loss:
      - Huber loss in log10(BER) space
      - Huber loss in raw BER space
      - optional regime weighting based on raw BER

    Critical stability fix:
      pred_log is CLAMPED BEFORE conversion to raw BER to avoid overflow in 10**pred_log.
    """
    def __init__(
        self,
        log_delta=0.5,
        raw_delta=0.01,
        alpha_log=0.9,
        beta_raw=0.1,
        use_regime_weights=False,
        low_thr=1e-6,
        mid_thr=1e-3,
        w_low=1.0,
        w_mid=1.25,
        w_high=1.75,
        min_log_ber=-12.0,
        max_log_ber=0.0,
    ):
        super().__init__()
        self.log_delta = log_delta
        self.raw_delta = raw_delta
        self.alpha_log = alpha_log
        self.beta_raw = beta_raw

        self.use_regime_weights = use_regime_weights
        self.low_thr = low_thr
        self.mid_thr = mid_thr
        self.w_low = w_low
        self.w_mid = w_mid
        self.w_high = w_high

        self.min_log_ber = min_log_ber
        self.max_log_ber = max_log_ber

    @staticmethod
    def huber_elementwise(pred, target, delta):
        err = pred - target
        abs_err = err.abs()
        return torch.where(
            abs_err < delta,
            0.5 * err * err,
            delta * (abs_err - 0.5 * delta)
        )

    def forward(self, pred_log, target_log):
        log_loss = self.huber_elementwise(pred_log, target_log, self.log_delta)

        pred_log_for_raw = pred_log.clamp(min=self.min_log_ber, max=self.max_log_ber)
        target_log_for_raw = target_log.clamp(min=self.min_log_ber, max=self.max_log_ber)

        pred_raw = torch.pow(10.0, pred_log_for_raw)
        target_raw = torch.pow(10.0, target_log_for_raw)

        raw_loss = self.huber_elementwise(pred_raw, target_raw, self.raw_delta)

        total = self.alpha_log * log_loss + self.beta_raw * raw_loss

        if self.use_regime_weights:
            weights = torch.full_like(target_raw, self.w_high)
            weights = torch.where(
                target_raw < self.mid_thr,
                torch.full_like(weights, self.w_mid),
                weights
            )
            weights = torch.where(
                target_raw < self.low_thr,
                torch.full_like(weights, self.w_low),
                weights
            )
            total = total * weights

        return total.mean()


# =========================
# MoE Head + Aux Losses
# =========================
class MoERegressionHead(nn.Module):
    """
    Soft mixture-of-experts head.

    Output:
      pred_log: [B, 1]
      gate_probs: [B, K]
      expert_preds: [B, K]
    """
    def __init__(self, in_dim, num_experts=3, gate_hidden=128, expert_hidden=(512, 256, 128), dropout=0.15):
        super().__init__()
        self.num_experts = num_experts

        self.gate = nn.Sequential(
            nn.Linear(in_dim, gate_hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(gate_hidden, num_experts)
        )

        experts = []
        for _ in range(num_experts):
            layers = []
            prev = in_dim
            for h in expert_hidden:
                layers.extend([
                    nn.Linear(prev, h),
                    nn.GELU(),
                    nn.Dropout(dropout),
                ])
                prev = h
            layers.append(nn.Linear(prev, 1))
            experts.append(nn.Sequential(*layers))
        self.experts = nn.ModuleList(experts)

    def forward(self, x):
        gate_logits = self.gate(x)                   # [B, K]
        gate_probs = torch.softmax(gate_logits, dim=1)

        expert_preds = torch.cat(
            [expert(x) for expert in self.experts], dim=1
        )                                            # [B, K]

        pred = torch.sum(gate_probs * expert_preds, dim=1, keepdim=True)
        return pred, gate_probs, expert_preds


def moe_balance_loss(gate_probs):
    """
    Encourage average routing to not collapse onto one expert.
    """
    num_experts = gate_probs.size(1)
    avg_usage = gate_probs.mean(dim=0)
    target = torch.full_like(avg_usage, 1.0 / num_experts)
    return torch.mean((avg_usage - target) ** 2)


# =========================
# Blocks
# =========================
class SqueezeExcite1D(nn.Module):
    def __init__(self, channels, reduction=8):
        super().__init__()
        hidden = max(channels // reduction, 8)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Sequential(
            nn.Conv1d(channels, hidden, kernel_size=1),
            nn.GELU(),
            nn.Conv1d(hidden, channels, kernel_size=1),
            nn.Sigmoid()
        )

    def forward(self, x):
        scale = self.pool(x)
        scale = self.fc(scale)
        return x * scale


class ConvBNAct(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size, dilation=1, stride=1, groups=1):
        super().__init__()
        padding = ((kernel_size - 1) // 2) * dilation
        self.block = nn.Sequential(
            nn.Conv1d(
                in_ch, out_ch,
                kernel_size=kernel_size,
                stride=stride,
                padding=padding,
                dilation=dilation,
                groups=groups,
                bias=False
            ),
            nn.BatchNorm1d(out_ch),
            nn.GELU()
        )

    def forward(self, x):
        return self.block(x)


class MultiScaleResidualBlock(nn.Module):
    def __init__(self, in_ch, out_ch, dropout=0.1, se=True):
        super().__init__()
        b1 = out_ch // 3
        b2 = out_ch // 3
        b3 = out_ch - b1 - b2

        self.branch1 = ConvBNAct(in_ch, b1, kernel_size=3, dilation=1)
        self.branch2 = ConvBNAct(in_ch, b2, kernel_size=5, dilation=1)
        self.branch3 = ConvBNAct(in_ch, b3, kernel_size=3, dilation=2)

        self.mix = nn.Sequential(
            nn.Conv1d(out_ch, out_ch, kernel_size=1, bias=False),
            nn.BatchNorm1d(out_ch),
            nn.GELU(),
            nn.Dropout(dropout)
        )

        self.se = SqueezeExcite1D(out_ch) if se else nn.Identity()

        self.skip = (
            nn.Sequential(
                nn.Conv1d(in_ch, out_ch, kernel_size=1, bias=False),
                nn.BatchNorm1d(out_ch)
            )
            if in_ch != out_ch else nn.Identity()
        )

        self.out_act = nn.GELU()

    def forward(self, x):
        residual = self.skip(x)
        out = torch.cat(
            [self.branch1(x), self.branch2(x), self.branch3(x)],
            dim=1
        )
        out = self.mix(out)
        out = self.se(out)
        out = out + residual
        out = self.out_act(out)
        return out


class DownsampleResidualBlock(nn.Module):
    def __init__(self, in_ch, out_ch, dropout=0.1, se=True):
        super().__init__()
        self.pre = MultiScaleResidualBlock(in_ch, out_ch, dropout=dropout, se=se)
        self.down = nn.Sequential(
            nn.Conv1d(
                out_ch, out_ch,
                kernel_size=3,
                stride=2,
                padding=1,
                bias=False
            ),
            nn.BatchNorm1d(out_ch),
            nn.GELU()
        )

    def forward(self, x):
        x = self.pre(x)
        x = self.down(x)
        return x


class ThresholdFiLM(nn.Module):
    def __init__(self, threshold_dim, feature_dim, hidden_dim=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(threshold_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, feature_dim * 2)
        )
        self.scale = nn.Parameter(torch.tensor(0.25, dtype=torch.float32))

    def forward(self, x, thr_emb):
        gamma_beta = self.net(thr_emb)
        gamma, beta = torch.chunk(gamma_beta, 2, dim=1)
        gamma = gamma.unsqueeze(-1)
        beta = beta.unsqueeze(-1)
        return x * (1.0 + self.scale * gamma) + self.scale * beta


class SequenceStem(nn.Module):
    def __init__(self, in_ch, stem_ch=32):
        super().__init__()
        self.net = nn.Sequential(
            ConvBNAct(in_ch, stem_ch, kernel_size=5),
            MultiScaleResidualBlock(stem_ch, stem_ch, dropout=0.05, se=True)
        )

    def forward(self, x):
        return self.net(x)


class LightSelfAttention1D(nn.Module):
    def __init__(self, dim, num_heads=4, mlp_ratio=2.0, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(
            embed_dim=dim,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True
        )
        self.norm2 = nn.LayerNorm(dim)

        hidden = int(dim * mlp_ratio)
        self.mlp = nn.Sequential(
            nn.Linear(dim, hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, dim),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        x_seq = x.transpose(1, 2)  # [B, L, C]
        y = self.norm1(x_seq)
        attn_out, _ = self.attn(y, y, y, need_weights=False)
        x_seq = x_seq + attn_out
        x_seq = x_seq + self.mlp(self.norm2(x_seq))
        return x_seq.transpose(1, 2)


class MultiHeadAttentionPooling(nn.Module):
    def __init__(self, input_dim, attn_dim=128, num_heads=4):
        super().__init__()
        self.score = nn.Sequential(
            nn.Conv1d(input_dim, attn_dim, kernel_size=1),
            nn.GELU(),
            nn.Conv1d(attn_dim, num_heads, kernel_size=1)
        )
        self.num_heads = num_heads
        self.input_dim = input_dim

    def forward(self, x):
        logits = self.score(x)
        weights = torch.softmax(logits, dim=-1)
        pooled = torch.einsum("bhl,bcl->bhc", weights, x)
        pooled = pooled.reshape(x.size(0), self.num_heads * self.input_dim)
        return pooled


class StagePooling(nn.Module):
    def __init__(self, channels, num_attn_heads=2, attn_dim=64):
        super().__init__()
        self.attn_pool = MultiHeadAttentionPooling(
            input_dim=channels,
            attn_dim=attn_dim,
            num_heads=num_attn_heads
        )
        self.out_dim = num_attn_heads * channels + 3 * channels

    def forward(self, x):
        attn = self.attn_pool(x)
        mean = x.mean(dim=-1)
        std = x.std(dim=-1, unbiased=False)
        maxv = x.amax(dim=-1)
        return torch.cat([attn, mean, std, maxv], dim=1)


# =========================
# Model
# =========================
class BERMultiScaleResCNNMoE(nn.Module):
    """
    Input seq channels:
      0 -> scaled tap means multiplied by number of molecules
      1 -> tap variances
      2 -> normalized position
      3 -> abs(scaled tap means)
      4 -> raw-ish log SNR proxy (scaled separately)
      5 -> first-position indicator
      6 -> z0 decision margin feature (repeated across sequence)
      7 -> z1 decision margin feature (repeated across sequence)

    Additional learned absolute positional embedding is injected in the model.

    Output:
      predicted log10(BER)
    """
    def __init__(
        self,
        seq_len,
        threshold_dim=1,
        stem_ch=32,
        channels=(64, 96, 128),
        dropout=0.10,
        stage_pool_heads=(2, 2, 2, 4),
        attn_heads=4,
        pos_emb_dim=8,
        use_self_attention=True,
        num_experts=3,
    ):
        super().__init__()

        self.seq_len = seq_len
        self.pos_emb_dim = pos_emb_dim
        self.use_self_attention = use_self_attention
        self.num_experts = num_experts

        self.mean_stem = SequenceStem(in_ch=1, stem_ch=stem_ch)
        self.var_stem = SequenceStem(in_ch=1, stem_ch=stem_ch)

        # extra_x contains:
        # [normalized position, abs tap, snr proxy, first flag, z0, z1]
        self.extra_proj = nn.Sequential(
            nn.Conv1d(6, stem_ch, kernel_size=1, bias=False),
            nn.BatchNorm1d(stem_ch),
            nn.GELU()
        )

        # learned absolute positional embedding: [1, pos_emb_dim, L]
        self.pos_embedding = nn.Parameter(
            torch.randn(1, pos_emb_dim, seq_len) * 0.02
        )

        self.pos_emb_proj = nn.Sequential(
            nn.Conv1d(pos_emb_dim, stem_ch, kernel_size=1, bias=False),
            nn.BatchNorm1d(stem_ch),
            nn.GELU()
        )

        fusion_in = stem_ch * 4

        self.thr_embed = nn.Sequential(
            nn.Linear(threshold_dim, 32),
            nn.GELU(),
            nn.Linear(32, 32),
            nn.GELU()
        )

        self.stage1 = MultiScaleResidualBlock(fusion_in, channels[0], dropout=dropout, se=True)
        self.film1 = ThresholdFiLM(32, channels[0], hidden_dim=64)

        self.stage2 = DownsampleResidualBlock(channels[0], channels[1], dropout=dropout, se=True)
        self.film2 = ThresholdFiLM(32, channels[1], hidden_dim=64)

        self.stage3 = DownsampleResidualBlock(channels[1], channels[2], dropout=dropout, se=True)
        self.film3 = ThresholdFiLM(32, channels[2], hidden_dim=64)

        if self.use_self_attention:
            self.self_attn = LightSelfAttention1D(
                dim=channels[2],
                num_heads=attn_heads,
                mlp_ratio=2.0,
                dropout=dropout
            )
        else:
            self.self_attn = nn.Identity()

        self.bottleneck = nn.Sequential(
            MultiScaleResidualBlock(channels[2], channels[2], dropout=dropout, se=True),
            MultiScaleResidualBlock(channels[2], channels[2], dropout=dropout, se=True),
        )

        self.pool1 = StagePooling(channels[0], num_attn_heads=stage_pool_heads[0], attn_dim=64)
        self.pool2 = StagePooling(channels[1], num_attn_heads=stage_pool_heads[1], attn_dim=64)
        self.pool3 = StagePooling(channels[2], num_attn_heads=stage_pool_heads[2], attn_dim=64)
        self.poolb = StagePooling(channels[2], num_attn_heads=stage_pool_heads[3], attn_dim=128)

        head_in = (
            self.pool1.out_dim +
            self.pool2.out_dim +
            self.pool3.out_dim +
            self.poolb.out_dim +
            32
        )

        self.moe_head = MoERegressionHead(
            in_dim=head_in,
            num_experts=num_experts,
            gate_hidden=128,
            expert_hidden=(512, 256, 128),
            dropout=0.15
        )

    def extract_fused_features(self, seq, threshold):
        mean_x = seq[:, 0:1, :]
        var_x = seq[:, 1:2, :]
        extra_x = seq[:, 2:8, :]  # pos, abs, snr, first_flag, z0, z1

        mean_f = self.mean_stem(mean_x)
        var_f = self.var_stem(var_x)
        extra_f = self.extra_proj(extra_x)

        pos_emb = self.pos_embedding.expand(seq.size(0), -1, -1)
        pos_f = self.pos_emb_proj(pos_emb)

        x = torch.cat([mean_f, var_f, extra_f, pos_f], dim=1)
        thr = self.thr_embed(threshold)

        x1 = self.stage1(x)
        x1 = self.film1(x1, thr)

        x2 = self.stage2(x1)
        x2 = self.film2(x2, thr)

        x3 = self.stage3(x2)
        x3 = self.film3(x3, thr)

        x3 = self.self_attn(x3)
        xb = self.bottleneck(x3)

        p1 = self.pool1(x1)
        p2 = self.pool2(x2)
        p3 = self.pool3(x3)
        pb = self.poolb(xb)

        fused = torch.cat([p1, p2, p3, pb, thr], dim=1)
        return fused

    def forward(self, seq, threshold, return_aux=False):
        fused = self.extract_fused_features(seq, threshold)
        pred_log, gate_probs, expert_preds = self.moe_head(fused)

        if return_aux:
            return pred_log, gate_probs, expert_preds
        return pred_log


# =========================
# Data
# =========================
def prepare_data(csv_path, batch_size=256, nrows=5000000, num_workers=0):
    df = pd.read_csv(csv_path, nrows=nrows)

    if "mem_len" not in df.columns:
        raise ValueError("Required column 'mem_len' not found.")
    if "N" not in df.columns:
        raise ValueError("Required column 'N' not found.")

    df = df[df["mem_len"] != 1].copy()

    tap_cols = get_sorted_seq_cols(df.columns, "tap")
    var_cols = get_sorted_seq_cols(df.columns, "var")

    if not tap_cols:
        raise ValueError("No tap_* columns found.")
    if not var_cols:
        raise ValueError("No var_* columns found.")
    if len(tap_cols) != len(var_cols):
        raise ValueError(
            f"tap/var length mismatch: {len(tap_cols)} tap cols vs {len(var_cols)} var cols"
        )

    required_cols = tap_cols + var_cols + ["threshold", "BER", "N"]
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    df[tap_cols] = df[tap_cols].fillna(0.0)
    df[var_cols] = df[var_cols].fillna(0.0)
    df["threshold"] = df["threshold"].fillna(0.0)
    df["BER"] = df["BER"].fillna(0.0)
    df["N"] = df["N"].fillna(0.0)

    df = df[(df["threshold"] > 0) & (df["BER"] > 0) & (df["N"] > 0)].copy()

    X_taps_raw = df[tap_cols].to_numpy(dtype=np.float32)
    X_vars_raw = df[var_cols].to_numpy(dtype=np.float32)
    num_molecules = df["N"].to_numpy(dtype=np.float32).reshape(-1, 1)

    X_thr_raw = df["threshold"].to_numpy(dtype=np.float32).reshape(-1, 1)
    y_raw = df["BER"].to_numpy(dtype=np.float32).reshape(-1, 1)

    X_thr = np.log10(X_thr_raw + EPS).astype(np.float32)
    y_log = np.log10(y_raw + EPS).astype(np.float32)

    # Mean is scaled to expected count-like value
    X_taps_feat = (X_taps_raw * num_molecules).astype(np.float32)

    if np.any(X_vars_raw < 0):
        raise ValueError("Variance columns contain negative values; cannot use them safely.")

    # Use variance directly (not std)
    X_vars_feat = X_vars_raw.astype(np.float32)

    abs_taps_raw = np.abs(X_taps_feat).astype(np.float32)

    # SNR proxy based on count-scaled mean and variance
    snr_raw = np.log10((X_taps_feat ** 2) / (X_vars_raw + EPS) + EPS).astype(np.float32)

    # -------------------------
    # Global decision features
    # -------------------------
    first_mean = X_taps_feat[:, 0:1]
    past_means = X_taps_feat[:, 1:]
    first_var = X_vars_feat[:, 0:1]
    past_vars = X_vars_feat[:, 1:]

    # Expected interference from past bits with P(bit=1)=0.5
    mu0_raw = 0.5 * np.sum(past_means, axis=1, keepdims=True).astype(np.float32)
    mu1_raw = (first_mean + mu0_raw).astype(np.float32)

    # Practical first-order approximation for aggregate conditional variances
    var0_raw = (0.5 * np.sum(past_vars, axis=1, keepdims=True)).astype(np.float32)
    var1_raw = (first_var + var0_raw).astype(np.float32)

    std0_raw = np.sqrt(np.maximum(var0_raw, EPS)).astype(np.float32)
    std1_raw = np.sqrt(np.maximum(var1_raw, EPS)).astype(np.float32)

    z0_raw = ((X_thr_raw - mu0_raw) / (std0_raw + EPS)).astype(np.float32)
    z1_raw = ((mu1_raw - X_thr_raw) / (std1_raw + EPS)).astype(np.float32)

    strat_labels = make_strat_bins(y_log, n_bins=10)

    split_args = dict(test_size=0.30, random_state=42)
    if strat_labels is not None:
        split_args["stratify"] = strat_labels

    (
        t_taps_raw, temp_taps_raw,
        t_vars_raw, temp_vars_raw,
        t_abs_raw, temp_abs_raw,
        t_snr_raw, temp_snr_raw,
        t_z0_raw, temp_z0_raw,
        t_z1_raw, temp_z1_raw,
        t_thr, temp_thr,
        t_y_log, temp_y_log
    ) = train_test_split(
        X_taps_feat, X_vars_feat, abs_taps_raw, snr_raw, z0_raw, z1_raw, X_thr, y_log,
        **split_args
    )

    temp_strat = make_strat_bins(temp_y_log, n_bins=6)

    split_args2 = dict(test_size=0.50, random_state=42)
    if temp_strat is not None:
        split_args2["stratify"] = temp_strat

    (
        v_taps_raw, te_taps_raw,
        v_vars_raw, te_vars_raw,
        v_abs_raw, te_abs_raw,
        v_snr_raw, te_snr_raw,
        v_z0_raw, te_z0_raw,
        v_z1_raw, te_z1_raw,
        v_thr, te_thr,
        v_y_log, te_y_log
    ) = train_test_split(
        temp_taps_raw, temp_vars_raw, temp_abs_raw, temp_snr_raw,
        temp_z0_raw, temp_z1_raw, temp_thr, temp_y_log,
        **split_args2
    )

    tap_scaler = StandardScaler()
    var_scaler = StandardScaler()
    abs_scaler = StandardScaler()
    snr_scaler = StandardScaler()
    z0_scaler = StandardScaler()
    z1_scaler = StandardScaler()
    thr_scaler = StandardScaler()

    t_taps = tap_scaler.fit_transform(t_taps_raw).astype(np.float32)
    v_taps = tap_scaler.transform(v_taps_raw).astype(np.float32)
    te_taps = tap_scaler.transform(te_taps_raw).astype(np.float32)

    t_vars = var_scaler.fit_transform(t_vars_raw).astype(np.float32)
    v_vars = var_scaler.transform(v_vars_raw).astype(np.float32)
    te_vars = var_scaler.transform(te_vars_raw).astype(np.float32)

    t_abs = abs_scaler.fit_transform(t_abs_raw).astype(np.float32)
    v_abs = abs_scaler.transform(v_abs_raw).astype(np.float32)
    te_abs = abs_scaler.transform(te_abs_raw).astype(np.float32)

    t_snr = snr_scaler.fit_transform(t_snr_raw).astype(np.float32)
    v_snr = snr_scaler.transform(v_snr_raw).astype(np.float32)
    te_snr = snr_scaler.transform(te_snr_raw).astype(np.float32)

    t_z0 = z0_scaler.fit_transform(t_z0_raw).astype(np.float32)
    v_z0 = z0_scaler.transform(v_z0_raw).astype(np.float32)
    te_z0 = z0_scaler.transform(te_z0_raw).astype(np.float32)

    t_z1 = z1_scaler.fit_transform(t_z1_raw).astype(np.float32)
    v_z1 = z1_scaler.transform(v_z1_raw).astype(np.float32)
    te_z1 = z1_scaler.transform(te_z1_raw).astype(np.float32)

    t_thr = thr_scaler.fit_transform(t_thr).astype(np.float32)
    v_thr = thr_scaler.transform(v_thr).astype(np.float32)
    te_thr = thr_scaler.transform(te_thr).astype(np.float32)

    L = t_taps.shape[1]
    pos = np.linspace(0.0, 1.0, L, dtype=np.float32)

    def build_features(taps_scaled, vars_scaled, abs_scaled, snr_scaled, z0_scaled, z1_scaled):
        n = taps_scaled.shape[0]

        pos_ch = np.tile(pos, (n, 1)).astype(np.float32)

        first_flag = np.zeros((n, L), dtype=np.float32)
        first_flag[:, 0] = 1.0

        z0_ch = np.tile(z0_scaled, (1, L)).astype(np.float32)
        z1_ch = np.tile(z1_scaled, (1, L)).astype(np.float32)

        seq = np.stack(
            [
                taps_scaled,
                vars_scaled,
                pos_ch,
                abs_scaled,
                snr_scaled,
                first_flag,
                z0_ch,
                z1_ch,
            ],
            axis=1
        ).astype(np.float32)
        return seq

    t_seq = build_features(t_taps, t_vars, t_abs, t_snr, t_z0, t_z1)
    v_seq = build_features(v_taps, v_vars, v_abs, v_snr, v_z0, v_z1)
    te_seq = build_features(te_taps, te_vars, te_abs, te_snr, te_z0, te_z1)

    train_ds = TensorDataset(
        torch.from_numpy(t_seq),
        torch.from_numpy(t_thr),
        torch.from_numpy(t_y_log)
    )
    val_ds = TensorDataset(
        torch.from_numpy(v_seq),
        torch.from_numpy(v_thr),
        torch.from_numpy(v_y_log)
    )
    test_ds = TensorDataset(
        torch.from_numpy(te_seq),
        torch.from_numpy(te_thr),
        torch.from_numpy(te_y_log)
    )

    pin_mem = torch.cuda.is_available()

    train_loader = DataLoader(
        train_ds,
        batch_size=batch_size,
        shuffle=True,
        pin_memory=pin_mem,
        num_workers=num_workers
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=batch_size,
        shuffle=False,
        pin_memory=pin_mem,
        num_workers=num_workers
    )
    test_loader = DataLoader(
        test_ds,
        batch_size=batch_size,
        shuffle=False,
        pin_memory=pin_mem,
        num_workers=num_workers
    )

    scalers = {
        "tap_scaler": tap_scaler,
        "var_scaler": var_scaler,
        "abs_scaler": abs_scaler,
        "snr_scaler": snr_scaler,
        "z0_scaler": z0_scaler,
        "z1_scaler": z1_scaler,
        "thr_scaler": thr_scaler,
        "tap_cols": tap_cols,
        "var_cols": var_cols,
        "seq_channels": 8,
        "feature_order": [
            "tap_scaled_mean",
            "variance",
            "pos",
            "abs_tap_scaled_mean",
            "snr_proxy_log",
            "first_flag",
            "z0_margin",
            "z1_margin",
        ],
        "seq_len": L,
        "uses_learned_positional_embedding": True,
        "pos_emb_dim": 8,
        "uses_self_attention": True,
        "uses_moe_head": True,
        "num_experts": 3,
        "preprocessing": {
            "tap_transform": "tap_mean * num_molecules",
            "var_transform": "use raw variance",
            "z0_definition": "z0 = (threshold_raw - mu0) / sqrt(var0 + eps)",
            "z1_definition": "z1 = (mu1 - threshold_raw) / sqrt(var1 + eps)",
            "mu0_definition": "0.5 * sum(past_scaled_means)",
            "mu1_definition": "first_scaled_mean + mu0",
            "var0_definition": "0.5 * sum(past_variances)",
            "var1_definition": "first_variance + var0",
        },
    }

    return train_loader, val_loader, test_loader, scalers, len(tap_cols)


# =========================
# Evaluation
# =========================
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    all_preds_log = []
    all_targets_log = []
    all_gate_probs = []

    with torch.no_grad():
        for b_seq, b_thr, b_y_log in loader:
            b_seq = b_seq.to(device, non_blocking=True)
            b_thr = b_thr.to(device, non_blocking=True)
            b_y_log = b_y_log.to(device, non_blocking=True)

            pred_log, gate_probs, _ = model(b_seq, b_thr, return_aux=True)

            if has_nonfinite_tensor(pred_log):
                raise RuntimeError("Non-finite prediction detected during evaluation.")

            loss = criterion(pred_log, b_y_log)

            if has_nonfinite_tensor(loss):
                raise RuntimeError("Non-finite loss detected during evaluation.")

            total_loss += loss.item()
            all_preds_log.append(pred_log.cpu().numpy())
            all_targets_log.append(b_y_log.cpu().numpy())
            all_gate_probs.append(gate_probs.cpu().numpy())

    avg_loss = total_loss / max(len(loader), 1)

    preds_log = np.vstack(all_preds_log)
    targets_log = np.vstack(all_targets_log)
    gate_probs = np.vstack(all_gate_probs)

    preds_raw = np.clip(10 ** np.clip(preds_log, -12.0, 0.0), EPS, 1.0)
    targets_raw = np.clip(10 ** np.clip(targets_log, -12.0, 0.0), EPS, 1.0)

    rmse_log = float(np.sqrt(np.mean((preds_log - targets_log) ** 2)))
    mae_log = float(np.mean(np.abs(preds_log - targets_log)))
    factor_error = float(10 ** rmse_log)

    rmse_raw = float(np.sqrt(np.mean((preds_raw - targets_raw) ** 2)))
    mae_raw = float(np.mean(np.abs(preds_raw - targets_raw)))

    avg_gate_usage = gate_probs.mean(axis=0)

    return avg_loss, rmse_log, mae_log, factor_error, rmse_raw, mae_raw, avg_gate_usage


def evaluate_by_target_range(model, loader, device):
    model.eval()
    all_preds_log = []
    all_targets_log = []
    all_gate_probs = []

    with torch.no_grad():
        for b_seq, b_thr, b_y_log in loader:
            b_seq = b_seq.to(device, non_blocking=True)
            b_thr = b_thr.to(device, non_blocking=True)
            pred_log, gate_probs, _ = model(b_seq, b_thr, return_aux=True)

            if has_nonfinite_tensor(pred_log):
                raise RuntimeError("Non-finite prediction detected during per-range evaluation.")

            all_preds_log.append(pred_log.cpu().numpy())
            all_targets_log.append(b_y_log.numpy())
            all_gate_probs.append(gate_probs.cpu().numpy())

    preds_log = np.vstack(all_preds_log).reshape(-1)
    targets_log = np.vstack(all_targets_log).reshape(-1)
    gate_probs = np.vstack(all_gate_probs)

    preds_raw = np.clip(10 ** np.clip(preds_log, -12.0, 0.0), EPS, 1.0)
    targets_raw = np.clip(10 ** np.clip(targets_log, -12.0, 0.0), EPS, 1.0)

    ranges = {
        "low_BER(y<1e-6)": targets_raw < 1e-6,
        "mid_BER(1e-6<=y<1e-3)": (targets_raw >= 1e-6) & (targets_raw < 1e-3),
        "high_BER(y>=1e-3)": targets_raw >= 1e-3,
    }

    metrics = {}
    for name, mask in ranges.items():
        if np.any(mask):
            rmse_log = float(np.sqrt(np.mean((preds_log[mask] - targets_log[mask]) ** 2)))
            mae_log = float(np.mean(np.abs(preds_log[mask] - targets_log[mask])))
            rmse_raw = float(np.sqrt(np.mean((preds_raw[mask] - targets_raw[mask]) ** 2)))
            mae_raw = float(np.mean(np.abs(preds_raw[mask] - targets_raw[mask])))
            gate_usage = gate_probs[mask].mean(axis=0)

            metrics[name] = {
                "count": int(mask.sum()),
                "rmse_log": rmse_log,
                "mae_log": mae_log,
                "factor_error": float(10 ** rmse_log),
                "rmse_raw": rmse_raw,
                "mae_raw": mae_raw,
                "avg_gate_usage": gate_usage.tolist(),
            }
        else:
            metrics[name] = None

    return metrics


# =========================
# Training
# =========================
def train_engine():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Executing on: {device}")

    train_loader, val_loader, test_loader, scalers, seq_len = prepare_data(
        DATA_PATH,
        batch_size=256,
        nrows=5000000,
        num_workers=0
    )

    joblib.dump(scalers, SCALER_SAVE_PATH)

    model = BERMultiScaleResCNNMoE(
        seq_len=seq_len,
        threshold_dim=1,
        stem_ch=32,
        channels=(64, 96, 128),
        dropout=0.10,
        stage_pool_heads=(2, 2, 2, 4),
        attn_heads=4,
        pos_emb_dim=8,
        use_self_attention=True,
        num_experts=3,
    ).to(device)

    optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)

    criterion = StableMultiObjectiveBERLoss(
        log_delta=0.5,
        raw_delta=0.01,
        alpha_log=0.9,
        beta_raw=0.1,
        use_regime_weights=False,
        low_thr=1e-6,
        mid_thr=1e-3,
        w_low=1.0,
        w_mid=1.25,
        w_high=1.75,
        min_log_ber=-12.0,
        max_log_ber=0.0,
    )

    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        patience=4,
        factor=0.5
    )

    # Auxiliary MoE loss weights
    lambda_gate_ce = 0.05
    lambda_balance = 0.01

    gate_ce_loss_fn = nn.NLLLoss()

    best_val_loss = float("inf")
    best_state = None
    patience = 12
    wait = 0
    min_epochs_before_early_stop = 60

    print(f"Detected sequence length L = {seq_len}")

    training_broke = False

    for epoch in range(100):
        model.train()

        running_main_loss = 0.0
        running_total_loss = 0.0
        running_gate_ce = 0.0
        running_balance = 0.0
        running_abs_err_log = 0.0
        running_sq_err_log = 0.0
        running_abs_err_raw = 0.0
        running_count = 0
        running_gate_usage_sum = torch.zeros(model.num_experts, dtype=torch.float64)

        for batch_idx, (b_seq, b_thr, b_y_log) in enumerate(train_loader):
            b_seq = b_seq.to(device, non_blocking=True)
            b_thr = b_thr.to(device, non_blocking=True)
            b_y_log = b_y_log.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            pred_log, gate_probs, _ = model(b_seq, b_thr, return_aux=True)

            if has_nonfinite_tensor(pred_log):
                print(f"Non-finite prediction detected at epoch {epoch+1}, batch {batch_idx+1}.")
                training_broke = True
                break

            main_loss = criterion(pred_log, b_y_log)
            if has_nonfinite_tensor(main_loss):
                print(f"Non-finite main loss detected at epoch {epoch+1}, batch {batch_idx+1}.")
                training_broke = True
                break

            regime_targets = log10ber_to_regime_index(b_y_log)
            gate_log_probs = torch.log(gate_probs.clamp_min(1e-8))
            gate_ce = gate_ce_loss_fn(gate_log_probs, regime_targets)

            balance = moe_balance_loss(gate_probs)

            loss = main_loss + lambda_gate_ce * gate_ce + lambda_balance * balance

            if has_nonfinite_tensor(loss):
                print(f"Non-finite total loss detected at epoch {epoch+1}, batch {batch_idx+1}.")
                training_broke = True
                break

            loss.backward()

            grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            if not torch.isfinite(grad_norm):
                print(f"Non-finite gradient norm detected at epoch {epoch+1}, batch {batch_idx+1}.")
                training_broke = True
                break

            optimizer.step()

            bad_param = False
            for name, param in model.named_parameters():
                if param.requires_grad and param.data is not None and not torch.isfinite(param.data).all():
                    print(f"Non-finite parameter detected after optimizer step: {name}")
                    bad_param = True
                    break
            if bad_param:
                training_broke = True
                break

            # Running training stats without full train-loader evaluation
            batch_size_actual = b_seq.size(0)
            running_main_loss += main_loss.item() * batch_size_actual
            running_total_loss += loss.item() * batch_size_actual
            running_gate_ce += gate_ce.item() * batch_size_actual
            running_balance += balance.item() * batch_size_actual

            pred_log_det = pred_log.detach()
            target_log_det = b_y_log.detach()

            abs_err_log = torch.abs(pred_log_det - target_log_det)
            sq_err_log = (pred_log_det - target_log_det) ** 2

            pred_raw = torch.pow(10.0, pred_log_det.clamp(-12.0, 0.0))
            target_raw = torch.pow(10.0, target_log_det.clamp(-12.0, 0.0))
            abs_err_raw = torch.abs(pred_raw - target_raw)

            running_abs_err_log += abs_err_log.sum().item()
            running_sq_err_log += sq_err_log.sum().item()
            running_abs_err_raw += abs_err_raw.sum().item()
            running_count += batch_size_actual

            running_gate_usage_sum += gate_probs.detach().sum(dim=0).cpu().double()

        if training_broke:
            print("Training stopped because non-finite values were detected.")
            break

        if running_count == 0:
            print("Training stopped because no valid batches were processed.")
            break

        train_main_loss = running_main_loss / running_count
        train_total_loss = running_total_loss / running_count
        train_gate_ce = running_gate_ce / running_count
        train_balance = running_balance / running_count
        train_mae_log = running_abs_err_log / running_count
        train_rmse_log = float(np.sqrt(running_sq_err_log / running_count))
        train_factor = float(10 ** train_rmse_log)
        train_mae_raw = running_abs_err_raw / running_count
        train_gate_usage = (running_gate_usage_sum / running_count).numpy()

        try:
            val_loss, val_rmse_log, val_mae_log, val_factor, val_rmse_raw, val_mae_raw, val_gate_usage = evaluate(
                model, val_loader, criterion, device
            )
        except RuntimeError as e:
            print(f"Evaluation failed at epoch {epoch+1}: {e}")
            break

        scheduler.step(val_loss)
        current_lr = optimizer.param_groups[0]["lr"]

        print(
            f"Epoch {epoch+1:03d} | "
            f"LR: {current_lr:.2e} | "
            f"Train Main Loss: {train_main_loss:.4f} | "
            f"Train Total Loss: {train_total_loss:.4f} | "
            f"Val Loss: {val_loss:.4f} | "
            f"Train RMSE(log10): {train_rmse_log:.4f} (~{train_factor:.2f}x) | "
            f"Val RMSE(log10): {val_rmse_log:.4f} (~{val_factor:.2f}x) | "
            f"Train MAE(log10): {train_mae_log:.4f} | "
            f"Val MAE(log10): {val_mae_log:.4f} | "
            f"Train MAE(raw): {train_mae_raw:.6f} | "
            f"Val MAE(raw): {val_mae_raw:.6f} | "
            f"Gate CE: {train_gate_ce:.4f} | "
            f"Balance: {train_balance:.6f}"
        )
        print(
            f"  Train gate usage: {np.round(train_gate_usage, 4).tolist()} | "
            f"Val gate usage: {np.round(val_gate_usage, 4).tolist()}"
        )

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())
            wait = 0
            print(f"  -> New best model saved at epoch {epoch+1} with Val Loss: {val_loss:.6f}")
        else:
            if epoch + 1 >= min_epochs_before_early_stop:
                wait += 1
                if wait >= patience:
                    print("Early stopping triggered.")
                    break

    torch.save(model.state_dict(), LAST_MODEL_SAVE_PATH)
    print(f"Last model saved to: {LAST_MODEL_SAVE_PATH}")

    if best_state is not None:
        model.load_state_dict(best_state)
        torch.save(model.state_dict(), BEST_MODEL_SAVE_PATH)
        print(f"Best model saved to: {BEST_MODEL_SAVE_PATH}")
    else:
        print("Warning: no valid best checkpoint was found.")

    test_loss, test_rmse_log, test_mae_log, test_factor, test_rmse_raw, test_mae_raw, test_gate_usage = evaluate(
        model, test_loader, criterion, device
    )

    print(
        f"Test Loss: {test_loss:.4f} | "
        f"Test RMSE(log10): {test_rmse_log:.4f} | "
        f"Test MAE(log10): {test_mae_log:.4f} | "
        f"Typical multiplicative error: ~{test_factor:.2f}x | "
        f"Test RMSE(raw): {test_rmse_raw:.6f} | "
        f"Test MAE(raw): {test_mae_raw:.6f}"
    )
    print(f"Test gate usage: {np.round(test_gate_usage, 4).tolist()}")

    range_metrics = evaluate_by_target_range(model, test_loader, device)
    print("\nPer-range test diagnostics:")
    for name, stats in range_metrics.items():
        if stats is None:
            print(f"{name}: no samples")
        else:
            print(
                f"{name} | count={stats['count']} | "
                f"RMSE(log10)={stats['rmse_log']:.4f} | "
                f"MAE(log10)={stats['mae_log']:.4f} | "
                f"factor~{stats['factor_error']:.2f}x | "
                f"RMSE(raw)={stats['rmse_raw']:.6f} | "
                f"MAE(raw)={stats['mae_raw']:.6f} | "
                f"gate_usage={np.round(stats['avg_gate_usage'], 4).tolist()}"
            )

    return model


if __name__ == "__main__":
    os.makedirs(BASE_DIR, exist_ok=True)

    if os.path.exists(DATA_PATH):
        print(f"Reading data from: {DATA_PATH}")
        trained_model = train_engine()
        print(f"Scalers saved to: {SCALER_SAVE_PATH}")
    else:
        print(f"Critical Error: Data file not found at {DATA_PATH}")

Reading data from: ./data_physics_with_variances_total.csv
Executing on: cuda
Detected sequence length L = 14
Epoch 001 | LR: 1.00e-04 | Train Main Loss: 0.1654 | Train Total Loss: 0.1731 | Val Loss: 0.0390 | Train RMSE(log10): 1.3062 (~20.24x) | Val RMSE(log10): 0.4999 (~3.16x) | Train MAE(log10): 0.4766 | Val MAE(log10): 0.1447 | Train MAE(raw): 0.031807 | Val MAE(raw): 0.013377 | Gate CE: 0.1344 | Balance: 0.100648
  Train gate usage: [0.1938, 0.0344, 0.7717] | Val gate usage: [0.18379999697208405, 0.04430000111460686, 0.7718999981880188]
  -> New best model saved at epoch 1 with Val Loss: 0.039002
Epoch 002 | LR: 1.00e-04 | Train Main Loss: 0.0835 | Train Total Loss: 0.0891 | Val Loss: 0.0545 | Train RMSE(log10): 0.7873 (~6.13x) | Val RMSE(log10): 0.6639 (~4.61x) | Train MAE(log10): 0.2753 | Val MAE(log10): 0.1930 | Train MAE(raw): 0.020056 | Val MAE(raw): 0.015666 | Gate CE: 0.0938 | Balance: 0.098162
  Train gate usage: [0.1776, 0.0527, 0.7698] | Val gate usage: [0.19320000708103

Epoch 019 | LR: 2.50e-05 | Train Main Loss: 0.0159 | Train Total Loss: 0.0190 | Val Loss: 0.0208 | Train RMSE(log10): 0.2155 (~1.64x) | Val RMSE(log10): 0.2820 (~1.91x) | Train MAE(log10): 0.0922 | Val MAE(log10): 0.0907 | Train MAE(raw): 0.007860 | Val MAE(raw): 0.008258 | Gate CE: 0.0430 | Balance: 0.100561
  Train gate usage: [0.1735, 0.0512, 0.7753] | Val gate usage: [0.1785999983549118, 0.05290000140666962, 0.7681999802589417]
Epoch 020 | LR: 2.50e-05 | Train Main Loss: 0.0154 | Train Total Loss: 0.0185 | Val Loss: 0.0136 | Train RMSE(log10): 0.2115 (~1.63x) | Val RMSE(log10): 0.2214 (~1.66x) | Train MAE(log10): 0.0907 | Val MAE(log10): 0.0692 | Train MAE(raw): 0.007781 | Val MAE(raw): 0.007059 | Gate CE: 0.0425 | Balance: 0.100553
  Train gate usage: [0.1733, 0.0514, 0.7753] | Val gate usage: [0.17599999904632568, 0.05249999836087227, 0.7710999846458435]
Epoch 021 | LR: 2.50e-05 | Train Main Loss: 0.0152 | Train Total Loss: 0.0183 | Val Loss: 0.0373 | Train RMSE(log10): 0.2097 (~

Epoch 038 | LR: 1.56e-06 | Train Main Loss: 0.0115 | Train Total Loss: 0.0144 | Val Loss: 0.0160 | Train RMSE(log10): 0.1715 (~1.48x) | Val RMSE(log10): 0.2280 (~1.69x) | Train MAE(log10): 0.0778 | Val MAE(log10): 0.0757 | Train MAE(raw): 0.007191 | Val MAE(raw): 0.006665 | Gate CE: 0.0374 | Balance: 0.100671
  Train gate usage: [0.1738, 0.0507, 0.7754] | Val gate usage: [0.1777999997138977, 0.052299998700618744, 0.7695000171661377]
Epoch 039 | LR: 1.56e-06 | Train Main Loss: 0.0113 | Train Total Loss: 0.0142 | Val Loss: 0.0364 | Train RMSE(log10): 0.1691 (~1.48x) | Val RMSE(log10): 0.4154 (~2.60x) | Train MAE(log10): 0.0772 | Val MAE(log10): 0.1313 | Train MAE(raw): 0.007174 | Val MAE(raw): 0.009346 | Gate CE: 0.0372 | Balance: 0.100666
  Train gate usage: [0.1739, 0.0507, 0.7754] | Val gate usage: [0.18170000612735748, 0.05339999869465828, 0.7645000219345093]
Epoch 040 | LR: 1.56e-06 | Train Main Loss: 0.0113 | Train Total Loss: 0.0142 | Val Loss: 0.0242 | Train RMSE(log10): 0.1697 (

Epoch 057 | LR: 1.95e-07 | Train Main Loss: 0.0111 | Train Total Loss: 0.0139 | Val Loss: 0.0168 | Train RMSE(log10): 0.1668 (~1.47x) | Val RMSE(log10): 0.2311 (~1.70x) | Train MAE(log10): 0.0764 | Val MAE(log10): 0.0781 | Train MAE(raw): 0.007153 | Val MAE(raw): 0.006945 | Gate CE: 0.0371 | Balance: 0.100682
  Train gate usage: [0.1737, 0.0508, 0.7755] | Val gate usage: [0.17749999463558197, 0.052799999713897705, 0.7692999839782715]
Epoch 058 | LR: 9.77e-08 | Train Main Loss: 0.0111 | Train Total Loss: 0.0140 | Val Loss: 0.0099 | Train RMSE(log10): 0.1668 (~1.47x) | Val RMSE(log10): 0.1710 (~1.48x) | Train MAE(log10): 0.0764 | Val MAE(log10): 0.0570 | Train MAE(raw): 0.007143 | Val MAE(raw): 0.006310 | Gate CE: 0.0371 | Balance: 0.100700
  Train gate usage: [0.1736, 0.0508, 0.7755] | Val gate usage: [0.17579999566078186, 0.052299998700618744, 0.7714999914169312]
Epoch 059 | LR: 9.77e-08 | Train Main Loss: 0.0111 | Train Total Loss: 0.0139 | Val Loss: 0.0219 | Train RMSE(log10): 0.1665